# Phase 1 Sanity Check: POSS-I-E Shadow Spot-Check

Recomputes Earth-umbra shadow geometry for sample POSS-I-E plates (both in-shadow and out-of-shadow) from `data/processed/exposures.parquet` and verifies `ObsE` start-time conversion and topocentric GEO parallax.

In [ ]:
import pandas as pd
from src.geometry import compute_exposure_shadow

df = pd.read_parquet('../data/processed/exposures.parquet')
poss = df[df['sub_archive'] == 'POSS-I-E'].copy()
print(f'POSS-I-E kept: {len(poss)}, any_in_shadow: {poss["any_in_shadow"].sum()}, fully_in_shadow: {poss["fully_in_shadow"].sum()}')

POSS-I-E kept: 914, any_in_shadow: 32, fully_in_shadow: 2


In [ ]:
in_sh = poss[poss['any_in_shadow']].sort_values('shadow_fraction', ascending=False).head(5)
out_sh = poss[~poss['any_in_shadow']].head(5)
spot = pd.concat([in_sh, out_sh])

for _, r in spot.iterrows():
    recomp = compute_exposure_shadow(
        ra_deg=r['ra_deg'],
        dec_deg=r['dec_deg'],
        width_deg=r['fov_ra_deg'],
        height_deg=r['fov_dec_deg'],
        mjd_start=r['mjd_start'],
        duration_sec=r['exptime_sec'],
        obs_lat_deg=r['obs_lat_deg'],
        obs_lon_deg=r['obs_lon_deg'],
        obs_elev_m=r['obs_elev_m'],
        altitudes_km=(42164.0,),
    )[42164.0]
    print(f"{r['exposure_id']:14s} stored={r['shadow_fraction']:.4f} recomputed={recomp:.4f} diff={abs(r['shadow_fraction']-recomp):.6f}")

POSSI_423_E    RA=110.10 Dec=+17.73 MJD_start=34386.27778 stored=1.0000 recomputed=1.0000 diff=0.000000
POSSI_823_E    RA=343.93 Dec=-17.23 MJD_start=34987.32014 stored=0.9993 recomputed=0.9993 diff=0.000000
POSSI_738_E    RA=193.90 Dec=-12.79 MJD_start=35571.28542 stored=0.9978 recomputed=0.9978 diff=0.000000
POSSI_1_E      RA=180.32 Dec=+89.72 MJD_start=34247.23056 stored=0.0000 recomputed=0.0000 diff=0.000000
POSSI_2_E      RA=180.32 Dec=+89.72 MJD_start=34246.35556 stored=0.0000 recomputed=0.0000 diff=0.000000
